In [3]:
"""
FASEROH: Transformer-based Symbolic Regression from Histogram Representations
Architecture: Conv1D -> Transformer Encoder -> Cross-Attention Pooling -> Autoregressive Decoder
Test function: 1 / (1 + sin(3x)), domain [0, 1]
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

# ─────────────────────────────────────────────
# 0.  Reproducibility
# ─────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────
# 1.  Vocabulary (tiny demo vocab)
# ─────────────────────────────────────────────
VOCAB = ["<pad>", "<sos>", "<eos>", "div", "add", "sin", "mul",
         "1", "3", "x", "<const>"]
VOCAB_SIZE   = len(VOCAB)
TOKEN2ID     = {t: i for i, t in enumerate(VOCAB)}
ID2TOKEN     = {i: t for t, i in TOKEN2ID.items()}

# Ground-truth prefix for  1 / (1 + sin(3x))
#   prefix: div 1 add 1 sin mul 3 x
GT_PREFIX = ["<sos>", "div", "1", "add", "1", "sin", "mul", "3", "x", "<eos>"]
GT_IDS    = [TOKEN2ID[t] for t in GT_PREFIX]

# ─────────────────────────────────────────────
# 2.  Data generation helpers
# ─────────────────────────────────────────────

def target_function(x: np.ndarray) -> np.ndarray:
    """f(x) = 1 / (1 + sin(3x))  on [0, 1]"""
    return 1.0 / (1.0 + np.sin(3.0 * x))


def build_histogram(n_samples: int, n_bins: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Sample n_samples points from f(x) over [0,1], treat f(x) values as a
    1-D 'density' distribution, and bin them into n_bins equal-width bins.

    Returns
    -------
    centers : (n_bins,)   bin centre positions  (c_k)
    densities: (n_bins,)  normalised density     (d_k)
    """
    x      = np.random.uniform(0, 1, n_samples)
    y      = target_function(x)                    # function values ≈ density proxy

    counts, edges = np.histogram(y, bins=n_bins, density=True)
    centers       = 0.5 * (edges[:-1] + edges[1:])
    densities     = counts / (counts.sum() + 1e-8)

    return centers.astype(np.float32), densities.astype(np.float32)


# ─────────────────────────────────────────────
# 3.  Positional Encoding
# ─────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe   = torch.zeros(max_len, d_model)
        pos  = torch.arange(0, max_len).unsqueeze(1).float()
        div  = torch.exp(torch.arange(0, d_model, 2).float()
                         * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model)"""
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ─────────────────────────────────────────────
# 4.  Histogram Encoder
# ─────────────────────────────────────────────

class HistogramEncoder(nn.Module):
    """
    Encodes an ordered sequence of histogram bins (c_k, d_k) into
    a fixed-size latent tensor via:
        Linear projection → Conv1D local features → PE → Transformer → Cross-Attn pooling
    """
    def __init__(
        self,
        d_model:      int = 128,
        n_heads:      int = 4,
        n_enc_layers: int = 3,
        n_latent:     int = 16,          # number of learnable query vectors (m)
        conv_channels:int = 128,
        conv_kernel:  int = 3,
        dropout:      float = 0.1,
    ):
        super().__init__()
        self.d_model   = d_model
        self.n_latent  = n_latent

        # ── (a) Input projection: R^2 → R^d ──────────────────────────────
        self.input_proj = nn.Linear(2, d_model)

        # ── (b) Conv1D local feature extractor ───────────────────────────
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, conv_channels, kernel_size=conv_kernel,
                      padding=conv_kernel // 2),
            nn.GELU(),
            nn.Conv1d(conv_channels, d_model, kernel_size=conv_kernel,
                      padding=conv_kernel // 2),
            nn.GELU(),
        )

        # ── (c) Positional encoding ───────────────────────────────────────
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # ── (d) Transformer encoder ───────────────────────────────────────
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer_enc = nn.TransformerEncoder(enc_layer, num_layers=n_enc_layers)

        # ── (e) Cross-attention pooling ───────────────────────────────────
        self.latent_queries = nn.Parameter(torch.randn(1, n_latent, d_model))
        self.cross_attn     = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )

    def forward(self, histogram: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        histogram : (B, K, 2)   stacked [c_k, d_k] for each bin k

        Returns
        -------
        Z : (B, m, d_model)   fixed-size latent representation
        """
        B, K, _ = histogram.shape
        print(f"\n{'='*55}")
        print(f"  ENCODER  |  Batch={B}, Bins K={K}, d_model={self.d_model}")
        print(f"{'='*55}")
        print(f"  [INPUT]        histogram          : {histogram.shape}")

        # (a) Linear projection
        E = self.input_proj(histogram)               # (B, K, d_model)
        print(f"  [PROJ]         after Linear(2→d)  : {E.shape}")

        # (b) Conv1D  (expects B, C, L)
        F_conv = self.conv(E.transpose(1, 2))        # (B, d_model, K)
        F_conv = F_conv.transpose(1, 2)              # (B, K, d_model)
        print(f"  [CONV1D]       after Conv1D        : {F_conv.shape}")

        # (c) Positional encoding
        F_pe   = self.pos_enc(F_conv)                # (B, K, d_model)
        print(f"  [POS_ENC]      after Pos Encoding  : {F_pe.shape}")

        # (d) Transformer encoder
        H_enc  = self.transformer_enc(F_pe)          # (B, K, d_model)
        print(f"  [TRANS_ENC]    after Transformer   : {H_enc.shape}")

        # (e) Cross-attention pooling
        queries = self.latent_queries.expand(B, -1, -1)   # (B, m, d_model)
        Z, _    = self.cross_attn(queries, H_enc, H_enc)  # (B, m, d_model)
        print(f"  [CROSS_POOL]   latent Z            : {Z.shape}  ← fixed size!")

        return Z   # (B, m, d_model)


# ─────────────────────────────────────────────
# 5.  Autoregressive Decoder
# ─────────────────────────────────────────────

class SymbolicDecoder(nn.Module):
    """
    Autoregressive transformer decoder.
    - Symbol head  : predicts next token  → cross-entropy loss
    - Constant head: predicts constant value when token == <const>
    """
    def __init__(
        self,
        vocab_size:    int,
        d_model:       int = 128,
        n_heads:       int = 4,
        n_dec_layers:  int = 3,
        dropout:       float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        # Token embedding + positional encoding
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=TOKEN2ID["<pad>"])
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)

        # Transformer decoder layers
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer_dec = nn.TransformerDecoder(dec_layer, num_layers=n_dec_layers)

        # Output heads
        self.symbol_head   = nn.Linear(d_model, vocab_size)
        self.constant_head = nn.Linear(d_model, 1)

    @staticmethod
    def _causal_mask(sz: int, device: torch.device) -> torch.Tensor:
        """Upper-triangular causal mask (True = masked)."""
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    def forward(
        self,
        tgt_ids: torch.Tensor,   # (B, T)  teacher-forced token ids
        memory:  torch.Tensor,   # (B, m, d_model)  encoder latent
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Returns
        -------
        symbol_logits : (B, T, vocab_size)
        const_preds   : (B, T, 1)
        """
        B, T = tgt_ids.shape
        print(f"\n{'='*55}")
        print(f"  DECODER  |  Batch={B}, SeqLen T={T}")
        print(f"{'='*55}")
        print(f"  [INPUT]        tgt_ids            : {tgt_ids.shape}")

        # Token embedding
        E = self.token_emb(tgt_ids)                  # (B, T, d_model)
        print(f"  [TOKEN_EMB]    after Embedding     : {E.shape}")

        # Positional encoding
        E = self.pos_enc(E)                          # (B, T, d_model)
        print(f"  [POS_ENC]      after Pos Encoding  : {E.shape}")

        # Causal mask
        causal = self._causal_mask(T, tgt_ids.device)

        # Transformer decoder (cross-attends to encoder memory)
        D = self.transformer_dec(
            tgt=E, memory=memory,
            tgt_mask=causal,
        )                                            # (B, T, d_model)
        print(f"  [TRANS_DEC]    after Transformer   : {D.shape}")

        # Dual output heads
        symbol_logits = self.symbol_head(D)          # (B, T, vocab_size)
        const_preds   = self.constant_head(D)        # (B, T, 1)
        print(f"  [SYMBOL_HEAD]  symbol logits       : {symbol_logits.shape}")
        print(f"  [CONST_HEAD]   constant preds      : {const_preds.shape}")

        return symbol_logits, const_preds


# ─────────────────────────────────────────────
# 6.  Full Model
# ─────────────────────────────────────────────

class FASeROH(nn.Module):
    """
    Full FASEROH pipeline:
        Histogram → HistogramEncoder → SymbolicDecoder → (symbol_logits, const_preds)
    """
    def __init__(
        self,
        vocab_size:    int   = VOCAB_SIZE,
        d_model:       int   = 128,
        n_heads:       int   = 4,
        n_enc_layers:  int   = 3,
        n_dec_layers:  int   = 3,
        n_latent:      int   = 16,
        dropout:       float = 0.1,
    ):
        super().__init__()
        self.encoder = HistogramEncoder(
            d_model=d_model, n_heads=n_heads,
            n_enc_layers=n_enc_layers, n_latent=n_latent,
            dropout=dropout,
        )
        self.decoder = SymbolicDecoder(
            vocab_size=vocab_size, d_model=d_model,
            n_heads=n_heads, n_dec_layers=n_dec_layers,
            dropout=dropout,
        )

    def forward(
        self,
        histogram: torch.Tensor,   # (B, K, 2)
        tgt_ids:   torch.Tensor,   # (B, T)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        Z             = self.encoder(histogram)
        logits, consts= self.decoder(tgt_ids, Z)
        return logits, consts


# ─────────────────────────────────────────────
# 7.  Loss Function
# ─────────────────────────────────────────────

def compute_loss(
    logits:  torch.Tensor,   # (B, T, vocab_size)
    consts:  torch.Tensor,   # (B, T, 1)
    tgt_ids: torch.Tensor,   # (B, T)
    const_id:int,
) -> torch.Tensor:
    B, T, V = logits.shape

    # Token classification loss  (shift: predict y_t from y_{<t})
    token_loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, V),
        tgt_ids[:, 1:].reshape(-1),
        ignore_index=TOKEN2ID["<pad>"],
    )

    # Constant regression loss  (only for <const> positions)
    const_mask = (tgt_ids[:, :-1] == const_id).float()   # (B, T-1)
    const_pred = consts[:, :-1, 0]                        # (B, T-1)
    const_loss = (const_mask * (const_pred ** 2)).sum() / (const_mask.sum() + 1e-8)

    return token_loss + 0.1 * const_loss


# ─────────────────────────────────────────────
# 8.  Test Run
# ─────────────────────────────────────────────

def test_run():
    print("\n" + "█"*55)
    print("  FASEROH  —  Architecture Test Run")
    print("  f(x) = 1 / (1 + sin(3x)),   x ∈ [0, 1]")
    print("█"*55)

    # ── Random n and K ────────────────────────────────────────
    n_samples = np.random.randint(500,  2001)   # number of data points
    n_bins    = np.random.randint(16,   65)     # number of histogram bins

    print(f"\n  [DATA]   n_samples = {n_samples},  n_bins (K) = {n_bins}")

    # ── Build histogram ───────────────────────────────────────
    centers, densities = build_histogram(n_samples, n_bins)

    print(f"  [HIST]   centers   shape : {centers.shape}")
    print(f"  [HIST]   densities shape : {densities.shape}")
    print(f"  [HIST]   centre range    : [{centers.min():.4f}, {centers.max():.4f}]")
    print(f"  [HIST]   density range   : [{densities.min():.4f}, {densities.max():.4f}]")

    # ── Batch = 1 tensor: (1, K, 2) ──────────────────────────
    hist_tensor = torch.tensor(
        np.stack([centers, densities], axis=-1)
    ).unsqueeze(0)                               # (1, K, 2)
    print(f"\n  [TENSOR] histogram tensor           : {hist_tensor.shape}  (B, K, 2)")

    # ── Target sequence (teacher-forced, B=1) ────────────────
    tgt_ids = torch.tensor([GT_IDS])             # (1, T)
    print(f"  [TENSOR] target token ids           : {tgt_ids.shape}  (B, T)")
    print(f"  [TOKENS] {GT_PREFIX}")

    # ── Build model ───────────────────────────────────────────
    model = FASeROH(
        vocab_size=VOCAB_SIZE,
        d_model=128,
        n_heads=4,
        n_enc_layers=3,
        n_dec_layers=3,
        n_latent=16,
        dropout=0.0,          # 0 for deterministic test
    )
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n  [MODEL]  Total parameters           : {total_params:,}")

    # ── Forward pass ─────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        logits, consts = model(hist_tensor, tgt_ids)

    # ── Loss (forward-only, no gradients) ────────────────────
    loss = compute_loss(logits, consts, tgt_ids, TOKEN2ID["<const>"])

    # ── Summary ──────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  FINAL SHAPES SUMMARY")
    print(f"{'='*55}")
    print(f"  histogram input   : {hist_tensor.shape}   (B, K, 2)")
    print(f"  symbol logits     : {logits.shape}         (B, T, vocab_size)")
    print(f"  constant preds    : {consts.shape}            (B, T, 1)")
    print(f"  loss (no grad)    : {loss.item():.4f}")

    # ── Predicted token sequence ─────────────────────────────
    pred_ids    = logits.argmax(dim=-1)[0].tolist()
    pred_tokens = [ID2TOKEN[i] for i in pred_ids]
    print(f"\n  [PRED]   predicted tokens : {pred_tokens}")
    print(f"  [GT]     ground truth     : {GT_PREFIX}")

    print("\n" + "█"*55)
    print("  Test run complete ✓")
    print("█"*55)



In [4]:
test_run()


███████████████████████████████████████████████████████
  FASEROH  —  Architecture Test Run
  f(x) = 1 / (1 + sin(3x)),   x ∈ [0, 1]
███████████████████████████████████████████████████████

  [DATA]   n_samples = 1626,  n_bins (K) = 44
  [HIST]   centers   shape : (44,)
  [HIST]   densities shape : (44,)
  [HIST]   centre range    : [0.5057, 0.9936]
  [HIST]   density range   : [0.0006, 0.2251]

  [TENSOR] histogram tensor           : torch.Size([1, 44, 2])  (B, K, 2)
  [TENSOR] target token ids           : torch.Size([1, 10])  (B, T)
  [TOKENS] ['<sos>', 'div', '1', 'add', '1', 'sin', 'mul', '3', 'x', '<eos>']

  [MODEL]  Total parameters           : 1,558,540

  ENCODER  |  Batch=1, Bins K=44, d_model=128
  [INPUT]        histogram          : torch.Size([1, 44, 2])
  [PROJ]         after Linear(2→d)  : torch.Size([1, 44, 128])
  [CONV1D]       after Conv1D        : torch.Size([1, 44, 128])
  [POS_ENC]      after Pos Encoding  : torch.Size([1, 44, 128])
  [TRANS_ENC]    after Transfo

/var/folders/fd/x4jg66r951xfppd52dj608zc0000gn/T/ipykernel_21815/2896253889.py:132: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_enc = nn.TransformerEncoder(enc_layer, num_layers=n_enc_layers)
